# Ternary Bonsai 2 4B — chat, vision, and tool calling on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Ternary-Bonsai-2-4B-llamacpp.ipynb)

[`prism-ml/Ternary-Bonsai-2-4B-gguf`](https://huggingface.co/prism-ml/Ternary-Bonsai-2-4B-gguf) is a Qwen3.5-4B vision-language model whose language weights are ternary (`-1, 0, +1`) with an FP16 scale per group. The `PQ2_0` pack is about 1.15 GB, down from about 8.4 GB at FP16, and the Q8_0 vision projector is another 0.37 GB. It still has the base model's 262K context, image input, and OpenAI-style tool calling.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. `PQ2_0` and `PTQ1_0` are custom GGUF types. Upstream llama.cpp only runs the `Q2_0` pack (build `b11555` or newer) and rejects the other two. This notebook uses the pack the card recommends, `PQ2_0`, so it needs the [PrismML llama.cpp fork](https://github.com/PrismML-Eng/llama.cpp). The binary is pinned to `prism-b10777-e3de715`, CUDA 12.8. Weights plus a short context land around 2–4 GB. The full 262K context is not allocated: these demos pass `-c 4096` for text and `-c 8192` for an image.

Thinking is on by default and can spend thousands of tokens before an answer. Every demo turns it off (`--reasoning off` and `enable_thinking: false`) and uses the card's non-thinking sampler: temperature 0.7, top_p 0.80, top_k 20, presence penalty 1.5, repeat penalty 1.0 over the last 64 tokens. `-fa on` is the card's flag; on a T4 (sm_75) the notebook passes `-fa off` because that GPU has no FlashAttention 2.

**Links:** [model card](https://huggingface.co/prism-ml/Ternary-Bonsai-2-4B-gguf) · [4B docs](https://docs.prismml.com/models/bonsai-2-4b) · [llama.cpp docs](https://docs.prismml.com/run/llamacpp)


## 1. Check the GPU


In [ ]:
import torch

!nvidia-smi

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

major, minor = torch.cuda.get_device_capability()
# The card passes -fa on. Turing (T4, sm_75) has no FlashAttention 2, so leave it off there.
FA = "on" if major >= 8 else "off"
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| -fa", FA)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


## 2. PrismML llama.cpp and the GGUF files

The CUDA 12.8 build looks for `libcudart.so.12`, which Colab ships under `/usr/local/cuda`. `PQ2_0` is the language model. `mmproj-Q8_0` is the vision tower the card recommends; the BF16 projector can crash `llama-server` when several image requests overlap.


In [ ]:
import glob
import os
import tarfile
import urllib.request

from huggingface_hub import hf_hub_download

PRISM_TAG = "prism-b10777-e3de715"
PRISM_URL = (
    "https://github.com/PrismML-Eng/llama.cpp/releases/download/"
    f"{PRISM_TAG}/llama-{PRISM_TAG}-bin-linux-cuda-12.8-x64.tar.gz"
)
BIN = "/content/bonsai-bin"
REPO = "prism-ml/Ternary-Bonsai-2-4B-gguf"


def cuda_lib_dirs():
    found = []
    for pattern in (
        "/usr/local/cuda*/lib64",
        "/usr/local/cuda*/targets/x86_64-linux/lib",
        "/usr/lib/x86_64-linux-gnu",
        "/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib",
    ):
        found.extend(glob.glob(pattern))
    return [path for path in found if os.path.isdir(path)]


if not os.path.exists(os.path.join(BIN, "llama-cli")):
    archive = "/tmp/bonsai-llama.tgz"
    urllib.request.urlretrieve(PRISM_URL, archive)
    os.makedirs(BIN, exist_ok=True)
    with tarfile.open(archive) as tar:
        tar.extractall(BIN)
    nested = [os.path.join(BIN, name) for name in os.listdir(BIN)]
    if len(nested) == 1 and os.path.isdir(nested[0]):
        for name in os.listdir(nested[0]):
            os.rename(os.path.join(nested[0], name), os.path.join(BIN, name))

libs = cuda_lib_dirs()
if not any(os.path.exists(os.path.join(path, "libcudart.so.12")) for path in [BIN, *libs]):
    raise SystemExit(
        "This PrismML CUDA 12.8 build needs libcudart.so.12 from the Colab CUDA toolkit "
        "(usually /usr/local/cuda/lib64). None was found."
    )

ENV = os.environ.copy()
ENV["LD_LIBRARY_PATH"] = ":".join([BIN, *libs, ENV.get("LD_LIBRARY_PATH", "")])
MODEL = hf_hub_download(REPO, "Ternary-Bonsai-2-4B-PQ2_0.gguf")
MMPROJ = hf_hub_download(REPO, "Ternary-Bonsai-2-4B-mmproj-Q8_0.gguf")
print(MODEL)
print(MMPROJ)

SAMPLER = [
    "--temp", "0.7",
    "--top-p", "0.8",
    "--top-k", "20",
    "--min-p", "0",
    "--presence-penalty", "1.5",
    "--repeat-penalty", "1.0",
    "--repeat-last-n", "64",
]


## 3. Chat

One short answer, thinking off, context capped at 4096. `--single-turn` makes `llama-cli` exit instead of waiting for another message.


In [ ]:
import subprocess

chat = subprocess.run(
    [
        os.path.join(BIN, "llama-cli"),
        "-m", MODEL,
        "-ngl", "99",
        "-fa", FA,
        "-c", "4096",
        "--jinja",
        "--reasoning", "off",
        "--chat-template-kwargs", '{"enable_thinking":false}',
        "--single-turn",
        "-n", "200",
        *SAMPLER,
        "-p", "Explain in three sentences why the sky is blue.",
    ],
    env=ENV,
    stdin=subprocess.DEVNULL,
    check=False,
)
print("llama-cli exit", chat.returncode)


## 4. Vision

The image is the one in the model card's server example. `llama-mtmd-cli` loads the Q8_0 projector only for this call. It is mirrored in this repo because upload.wikimedia.org often returns 429 to Colab ([dice by Ed g2s and Daniel G.](https://commons.wikimedia.org/wiki/File:PNG_transparency_demonstration_1.png), CC BY-SA 3.0).

In [ ]:
import requests
from IPython.display import Image, display

photo = "photo.png"
urls = [
    "https://raw.githubusercontent.com/donvito/notebooks/main/colab/data/ternary-bonsai/png-transparency-demonstration-1.png",
    "https://upload.wikimedia.org/wikipedia/commons/4/47/PNG_transparency_demonstration_1.png",
]
for url in urls:
    response = requests.get(url, headers={"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}, timeout=60)
    if response.ok:
        break
    print(f"Could not fetch {url}: HTTP {response.status_code}")
response.raise_for_status()
open(photo, "wb").write(response.content)
display(Image(filename=photo, width=360))

vision = subprocess.run(
    [
        os.path.join(BIN, "llama-mtmd-cli"),
        "-m", MODEL,
        "--mmproj", MMPROJ,
        "-ngl", "99",
        "-fa", FA,
        "-c", "8192",
        "--jinja",
        "--reasoning", "off",
        "--chat-template-kwargs", '{"enable_thinking":false}',
        "-n", "200",
        *SAMPLER,
        "--image", photo,
        "-p", "Describe this image in two sentences.",
    ],
    env=ENV,
    stdin=subprocess.DEVNULL,
    check=False,
)
print("llama-mtmd-cli exit", vision.returncode)


## 5. Tool calling

The 4B docs send a `tools` array to `llama-server` and read `tool_calls` back. `--jinja` is what turns that into native tool calls. The card asks for exactly one system message, and it has to be first. Tool calls on this family are occasionally malformed; the cell prints the raw message either way, then runs the function if the call parses.


In [ ]:
import json
import time

import requests

PORT = 8088
log_path = "/tmp/bonsai-server.log"
log = open(log_path, "w")
server = subprocess.Popen(
    [
        os.path.join(BIN, "llama-server"),
        "-m", MODEL,
        "-ngl", "99",
        "-fa", FA,
        "-c", "4096",
        "--jinja",
        "--reasoning", "off",
        "--host", "127.0.0.1",
        "--port", str(PORT),
        *SAMPLER,
    ],
    env=ENV,
    stdout=log,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)

base = f"http://127.0.0.1:{PORT}"
ready = False
for _ in range(60):
    if server.poll() is not None:
        break
    try:
        if requests.get(base + "/health", timeout=2).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        time.sleep(2)

if not ready:
    print(open(log_path).read()[-4000:])
    server.kill()
    raise SystemExit("llama-server did not become healthy. The log is above.")

tools = [{
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Multiply two numbers and return the product.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number"},
                "b": {"type": "number"},
            },
            "required": ["a", "b"],
        },
    },
}]
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 17 times 23? Use the multiply tool."},
]
payload = {
    "messages": messages,
    "tools": tools,
    "tool_choice": "auto",
    "chat_template_kwargs": {"enable_thinking": False},
    "max_tokens": 200,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "min_p": 0,
    "presence_penalty": 1.5,
    "repeat_penalty": 1.0,
}
first = requests.post(base + "/v1/chat/completions", json=payload, timeout=180)
first.raise_for_status()
message = first.json()["choices"][0]["message"]
print(json.dumps(message, indent=2)[:4000])
# The 4B template can attach a reasoning trace. Don't send it back into the next turn.
for key in ("reasoning_content", "reasoning"):
    message.pop(key, None)

calls = message.get("tool_calls") or []
if calls:
    call = calls[0]
    try:
        raw_args = call.get("function", {}).get("arguments") or "{}"
        args = json.loads(raw_args) if isinstance(raw_args, str) else raw_args
        product = float(args["a"]) * float(args["b"])
    except (KeyError, TypeError, json.JSONDecodeError, ValueError) as exc:
        print("Tool call did not parse:", exc)
        product = None
    if product is not None:
        print("tool result", product)
        follow = requests.post(
            base + "/v1/chat/completions",
            json={
                **payload,
                "messages": [
                    *messages,
                    message,
                    {
                        "role": "tool",
                        "tool_call_id": call.get("id", "call_1"),
                        "content": str(product),
                    },
                ],
            },
            timeout=180,
        )
        follow.raise_for_status()
        print(follow.json()["choices"][0]["message"].get("content"))
else:
    print("No tool_calls field. The model answered in text, or the call was malformed.")

server.terminate()
try:
    server.wait(timeout=10)
except subprocess.TimeoutExpired:
    server.kill()
log.close()
print("server stopped")
